In [ ]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import time

# ------------------------------------------------
# 0. 유틸리티 : 위키피디아에서 티커 목록 가져오기
# ------------------------------------------------
def get_sp500_symbols():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    table = pd.read_html(url, match="Symbol")[0]
    return table["Symbol"].tolist()

def get_nasdaq100_symbols():
    url = "https://en.wikipedia.org/wiki/Nasdaq-100"
    # 여러 표 중 'Ticker' 열이 있는 첫 번째 표 선택
    for tbl in pd.read_html(url):
        col_names = [str(c).lower() for c in tbl.columns]
        if any("ticker" in c for c in col_names):
            return tbl[tbl.columns[col_names.index("ticker")]].tolist()
    return []

# ------------------------------------------------
# 1. 스크리닝 대상 Universe
# ------------------------------------------------
sp500      = get_sp500_symbols()
nasdaq100  = get_nasdaq100_symbols()
universe   = sorted(set(sp500 + nasdaq100))

# ------------------------------------------------
# 2. 날짜 계산
# ------------------------------------------------
today      = datetime.utcnow()
one_month  = today - timedelta(days=30)

# ------------------------------------------------
# 3. 필터링
# ------------------------------------------------
passed = []

for tkr in universe:
    try:
        tic = yf.Ticker(tkr)

        # ------- (1) 분기별 매출·영업이익 QoQ 20% ↑ -------
        q_is = tic.quarterly_income_stmt
        if q_is is None or q_is.shape[1] < 2:
            continue
        q_is = q_is.loc[:, q_is.columns.sort_values(ascending=False)]  # 최근→과거
        # 라벨 호환성 확보
        rev_row = next((idx for idx in q_is.index if "revenue" in idx.lower()), None)
        op_row  = next((idx for idx in q_is.index if "operating" in idx.lower()
                                                and "income"   in idx.lower()), None)
        if not rev_row or not op_row:
            continue
        rev    = q_is.loc[rev_row].iloc[:2]
        op_inc = q_is.loc[op_row ].iloc[:2]
        rev_g  = (rev.iloc[0] - rev.iloc[1]) / abs(rev.iloc[1])
        op_g   = (op_inc.iloc[0] - op_inc.iloc[1]) / abs(op_inc.iloc[1])
        if rev_g < 0.20 or op_g < 0.20:
            continue

        # ------- (2) 사업설명에 '배터리 재활용' 키워드 -------
        summary = (tic.info.get("longBusinessSummary") or "").lower()
        if "battery" not in summary or "recycl" not in summary:
            continue

        # ------- (3) PER ≤ 30, PEG ≤ 1 -------
        pe  = tic.info.get("trailingPE")
        peg = tic.info.get("pegRatio")
        if pe is None or peg is None or pe > 20 or peg > 1:
            continue

        # ------- (4) 최근 30일 뉴스 ≥ 3개 -------
        news_items   = tic.news or []
        recent_news  = [n for n in news_items
                        if "providerPublishTime" in n and
                           datetime.utcfromtimestamp(n["providerPublishTime"]) >= one_month]
        if len(recent_news) < 0:
            continue

        # ------- 통과 종목 저장 -------
        passed.append({
            "Ticker":     tkr,
            "Rev QoQ %":  round(rev_g*100, 1),
            "OpInc QoQ %":round(op_g*100, 1),
            "PE":         pe,
            "PEG":        peg,
            "News(30d)":  len(recent_news)
        })

        time.sleep(0.15)   # 과도한 호출 방지용 (선택)

    except Exception:      # 데이터 누락·네트워크 오류 시 무시
        continue

# ------------------------------------------------
# 4. 결과 출력
# ------------------------------------------------
df = pd.DataFrame(passed)
print(df)


if df.empty:
    print("조건을 만족하는 종목이 없습니다.")
else:
    df = df.sort_values("PE")   # 'PE' 컬럼이 확실히 존재
    print(df.to_string(index=False))

print(f"\n조건 통과 종목 수: {len(df)}")
print(df.to_string(index=False))


Empty DataFrame
Columns: []
Index: []
조건을 만족하는 종목이 없습니다.

조건 통과 종목 수: 0
Empty DataFrame
Columns: []
Index: []
